In [ ]:
from dataclasses import dataclass
import time

import numpy as np
import primme
from scipy.sparse.linalg import LinearOperator

import libdet
from pyscf import ao2mo, gto, scf

In [2]:
def davidson_primme(
    ham,
    dets: np.ndarray,
    guess: np.ndarray | None = None,
    *,
    tol: float = 1e-8,
    max_iter: int = 200,
    max_space: int = 64,
    mode: str = "sparse",
):
    t0 = time.perf_counter()

    dets = libdet.to_dets(dets)
    hdiag = np.asarray(ham.diags(dets), dtype=np.float64).reshape(-1)
    n = hdiag.size

    if n == 1:
        t_total = time.perf_counter() - t0
        return (
            float(hdiag[0]),
            np.array([1.0], dtype=np.float64),
            hdiag,
            0.0,
            0.0,
            t_total,
        )

    if guess is None:
        v0 = np.zeros(n, dtype=np.float64)
        v0[0] = 1.0
    else:
        v0 = np.asarray(guess, dtype=np.float64).reshape(-1).copy()
        v0 /= np.linalg.norm(v0)

    t_mat = time.perf_counter()

    if mode == "sparse":
        A = ham.matrix(dets)
    elif mode == "matvec":
        def _matvec(x):
            x = np.asarray(x, dtype=np.float64).reshape(-1)
            return np.asarray(ham.matvec(dets, x, kets=dets), dtype=np.float64)

        def _matmat(X):
            X = np.asarray(X, dtype=np.float64)
            return np.asarray(ham.matvec(dets, X, kets=dets), dtype=np.float64)

        A = LinearOperator(
            shape=(n, n),
            matvec=_matvec,
            matmat=_matmat,
            dtype=np.float64,
        )
    else:
        raise ValueError("mode must be 'sparse' or 'matvec'")

    t_mat = time.perf_counter() - t_mat

    def _precond(x):
        X = np.asarray(x, dtype=np.float64)
        is_vec = X.ndim == 1
        if is_vec:
            X = X[:, None]

        shifts = np.asarray(
            primme.get_eigsh_param("ShiftsForPreconditioner"),
            dtype=np.float64,
        )
        if shifts.size == 0:
            shifts = np.zeros(X.shape[1], dtype=np.float64)
        elif shifts.size == 1 and X.shape[1] > 1:
            shifts = np.full(X.shape[1], shifts[0], dtype=np.float64)

        Y = np.empty_like(X)
        for j in range(X.shape[1]):
            denom = hdiag - shifts[j]
            denom = np.where(
                np.abs(denom) < 1e-8,
                np.copysign(1e-8, denom),
                denom,
            )
            Y[:, j] = X[:, j] / denom

        return Y[:, 0] if is_vec else Y

    OPinv = LinearOperator(
        shape=(n, n),
        matvec=_precond,
        matmat=_precond,
        dtype=np.float64,
    )

    ncv = min(n, max(8, min(max_space, 24)))

    t_solve = time.perf_counter()
    w, v = primme.eigsh(
        A,
        k=1,
        which="SA",
        v0=v0[:, None],
        OPinv=OPinv,
        tol=tol,
        maxiter=max_iter,
        ncv=ncv,
        maxBlockSize=1,
        raise_for_unconverged=False,
    )
    t_solve = time.perf_counter() - t_solve

    w = np.asarray(w, dtype=np.float64).reshape(-1)
    v = np.asarray(v, dtype=np.float64)

    if w.size == 0:
        c = v0.copy()
        Ac = A @ c
        e = float(np.dot(c, Ac))
        t_other = time.perf_counter() - t0 - t_mat - t_solve
        return e, c, hdiag, t_mat, t_solve, t_other

    c = v[:, 0].copy()
    if c[np.argmax(np.abs(c))] < 0.0:
        c = -c
    c /= np.linalg.norm(c)

    t_other = time.perf_counter() - t0 - t_mat - t_solve
    return float(w[0]), c, hdiag, t_mat, t_solve, t_other

In [3]:
@dataclass(slots=True)
class State:
    dets: np.ndarray
    coeffs: np.ndarray
    energy: float
    diags: np.ndarray
    eps: float | None = None


def hf_det(norb: int, nelec: tuple[int, int]) -> np.ndarray:
    nword = (norb + 63) // 64
    det = np.zeros((2, nword), dtype=np.uint64)

    for spin, nocc in enumerate(nelec):
        for p in range(nocc):
            det[spin, p // 64] |= np.uint64(1) << np.uint64(p % 64)

    return det


def hci_solve(
    ham,
    nelec: tuple[int, int],
    *,
    eps: float = 1e-4,
    max_cycle: int = 10,
) -> State:
    total_start = time.perf_counter()

    det0 = hf_det(int(ham.norb), nelec)
    dets = det0[np.newaxis]
    coeffs = np.array([1.0], dtype=np.float64)
    diags = ham.diags(dets)
    energy = float(diags[0])

    header = (
        f"{'Iter':>4} | {'Ndet':>8} | {'Screen (s)':>12} | "
        f"{'Matrix (s)':>12} | {'Solve (s)':>11} | {'OI (s)':>8} | {'Energy':>16}"
    )
    print(header)
    print("-" * len(header))

    for i in range(max_cycle):
        t_screen = time.perf_counter()

        cand_bras = ham.expand(
            dets,
            eps,
            coeffs=coeffs,
            exclude=dets,
        )
        t_screen = time.perf_counter() - t_screen

        n_old = len(dets)
        if cand_bras.shape[0] == 0:
            print(f"Converged at cycle {i}: no new determinants.")
            break

        trial_dets = np.concatenate([dets, cand_bras], axis=0)

        guess = np.zeros(trial_dets.shape[0], dtype=np.float64)
        guess[:n_old] = coeffs

        dets = np.ascontiguousarray(trial_dets, dtype=np.uint64)
        energy, coeffs, diags, t_mat, t_solve, t_other = davidson_primme(
            ham,
            dets,
            guess,
        )

        print(
            f"{i + 1:4d} | {len(dets):8d} | {t_screen:12.4f} | "
            f"{t_mat:12.4f} | {t_solve:11.4f} | {t_other:8.4f} | {energy:16.10f}"
        )

    print("-" * len(header))
    print(f"Total: {time.perf_counter() - total_start:.4f} s")

    return State(dets=dets, coeffs=coeffs, energy=energy, diags=diags, eps=float(eps))

In [4]:
"""
t_graph: time to build sparse H
t_solve: time for PRIMME Davidson
t_other: remaining overhead
"""

mol = gto.M(
    atom=
    '''
    O   0.00000000,  0.00000000,  0.00000000
    H   0.75700000,  0.00000000,  0.58590000
    H  -0.75700000,  0.00000000,  0.58590000
    ''',
    # atom=
    # '''
    # N   0.53920000,  0.00000000,  0.0000000
    # N   -0.539200000,  0.00000000,  0.0000000
    # ''',
    basis="cc-pvdz",
    unit="Angstrom",
    verbose=0,
)
mf = scf.RHF(mol).run()

norb = mf.mo_coeff.shape[1]
nelec = mol.nelec
h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
eri = ao2mo.restore(8, ao2mo.kernel(mol, mf.mo_coeff), norb)

ham = libdet.Hamiltonian.rhf(h1e, eri, ecore=mol.energy_nuc())
state = hci_solve(ham, nelec, eps=1e-4)

print(f"SCF      : {mf.e_tot:.12f}")
print(f"HCI var  : {state.energy:.12f}  Ndet: {len(state.dets)}")

Iter |     Ndet |   Screen (s) |   Matrix (s) |   Solve (s) |   OI (s) |           Energy
-----------------------------------------------------------------------------------------
   1 |     3270 |       0.0104 |       0.0365 |      0.0266 |   0.0004 |   -76.2312860243
   2 |   217996 |       0.1173 |       6.6447 |      1.9354 |   0.0176 |   -76.2431228756
   3 |   233367 |       0.4871 |       6.6568 |      1.7922 |   0.0379 |   -76.2432054888
   4 |   233573 |       0.4254 |       6.2610 |      1.3172 |   0.0184 |   -76.2432070671
   5 |   233575 |       0.3815 |       6.8748 |      1.0582 |   0.0208 |   -76.2432070861
Converged at cycle 5: no new determinants.
-----------------------------------------------------------------------------------------
Total: 34.6133 s
SCF      : -76.026796341400
HCI var  : -76.243207086118  Ndet: 233575


In [ ]:
# neural_external_demo.py
#
# Minimal validation script:
#   fixed P-space HCI state
#   MLP A_theta(D) for external amplitudes
#   Q-projected Schrodinger residual loss
#
# Assumes you already have:
#   ham   : libdet.Hamiltonian
#   state : State(dets, coeffs, energy, diags, eps)
#
# Example:
#   state = hci_solve(ham, nelec, eps=1e-4)
#   train_neural_external(ham, state)

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import jax
import jax.numpy as jnp

import libdet


# -----------------------------
# Determinant utilities
# -----------------------------

def det_key(det: np.ndarray) -> tuple[int, ...]:
    return tuple(int(x) for x in np.asarray(det, dtype=np.uint64).reshape(-1))


def unique_dets(*batches: np.ndarray) -> np.ndarray:
    """"Order-preserving unique determinant concatenation."""
    out = []
    seen = set()

    for batch in batches:
        batch = libdet.to_dets(batch)
        for det in batch:
            key = det_key(det)
            if key not in seen:
                seen.add(key)
                out.append(det.copy())

    if not out:
        raise ValueError("unique_dets received no determinants")

    return np.ascontiguousarray(np.stack(out, axis=0), dtype=np.uint64)


def coeff_lookup(dets: np.ndarray, p_coeffs: dict[tuple[int, ...], float]) -> tuple[np.ndarray, np.ndarray]:
    """Return is_p mask and P coefficient values for dets."""
    dets = libdet.to_dets(dets)

    is_p = np.zeros(dets.shape[0], dtype=bool)
    coeff = np.zeros(dets.shape[0], dtype=np.float64)

    for i, det in enumerate(dets):
        key = det_key(det)
        if key in p_coeffs:
            is_p[i] = True
            coeff[i] = p_coeffs[key]

    return is_p, coeff


def det_features(dets: np.ndarray, norb: int) -> np.ndarray:
    """Pure occupation-bit input: alpha occupations followed by beta occupations."""
    dets = libdet.to_dets(dets)
    n = dets.shape[0]

    x = np.zeros((n, 2 * norb), dtype=np.float32)

    for spin in range(2):
        for p in range(norb):
            word = p // 64
            bit = p % 64
            x[:, spin * norb + p] = ((dets[:, spin, word] >> np.uint64(bit)) & np.uint64(1)).astype(np.float32)

    return x


# -----------------------------
# Tiny MLP in pure JAX
# -----------------------------

def init_mlp(key: jax.Array, din: int, hidden: int = 64, depth: int = 2):
    keys = jax.random.split(key, depth + 1)
    params = []

    d0 = din
    for l in range(depth):
        w = 0.05 * jax.random.normal(keys[l], (d0, hidden), dtype=jnp.float32)
        b = jnp.zeros((hidden,), dtype=jnp.float32)
        params.append((w, b))
        d0 = hidden

    w = 0.01 * jax.random.normal(keys[-1], (d0, 1), dtype=jnp.float32)
    b = jnp.zeros((1,), dtype=jnp.float32)
    params.append((w, b))

    return params


def mlp_apply(params, x: jax.Array) -> jax.Array:
    h = x
    for w, b in params[:-1]:
        h = jnp.tanh(h @ w + b)

    w, b = params[-1]
    return (h @ w + b).reshape(-1)


@dataclass
class AdamState:
    m: object
    v: object
    t: int = 0


def tree_zeros_like(tree):
    return jax.tree_util.tree_map(jnp.zeros_like, tree)


def adam_init(params) -> AdamState:
    return AdamState(m=tree_zeros_like(params), v=tree_zeros_like(params), t=0)


def adam_step(params, grads, opt: AdamState, lr: float = 1e-3) -> tuple[object, AdamState]:
    b1 = 0.9
    b2 = 0.999
    eps = 1e-8
    t = opt.t + 1

    m = jax.tree_util.tree_map(lambda m, g: b1 * m + (1.0 - b1) * g, opt.m, grads)
    v = jax.tree_util.tree_map(lambda v, g: b2 * v + (1.0 - b2) * (g * g), opt.v, grads)

    mhat = jax.tree_util.tree_map(lambda x: x / (1.0 - b1**t), m)
    vhat = jax.tree_util.tree_map(lambda x: x / (1.0 - b2**t), v)

    params = jax.tree_util.tree_map(
        lambda p, mh, vh: p - lr * mh / (jnp.sqrt(vh) + eps),
        params,
        mhat,
        vhat,
    )

    return params, AdamState(m=m, v=v, t=t)


# -----------------------------
# libdet -> JAX residual batch
# -----------------------------

@dataclass
class ResidualBatch:
    h_rows_cols: jax.Array
    x_rows: jax.Array
    x_cols: jax.Array
    rows_is_p: jax.Array
    cols_is_p: jax.Array
    rows_c: jax.Array
    cols_c: jax.Array
    energy: float


def make_q_residual_batch(
    ham,
    p_dets: np.ndarray,
    p_coeffs_vec: np.ndarray,
    q_rows: np.ndarray,
    *,
    eps_qq: float,
) -> ResidualBatch:
    """Build one sampled Q-residual stencil.

    rows = sampled Q determinants
    cols = P union rows union screened neighbors of rows

    The residual is:
        r_a = H[a, cols] psi(cols) - E psi(a)
    for sampled a in Q.
    """
    p_dets = libdet.to_dets(p_dets)
    q_rows = libdet.to_dets(q_rows)

    conns = ham.conns(q_rows, eps_qq)
    q_neighbors = np.array(conns.bras, dtype=np.uint64, copy=True)

    rows = q_rows
    cols = unique_dets(p_dets, q_rows, q_neighbors)

    H = ham.matrix(rows, cols).toarray().astype(np.float32)

    p_coeffs = {
        det_key(det): float(c)
        for det, c in zip(p_dets, np.asarray(p_coeffs_vec, dtype=np.float64))
    }

    rows_is_p, rows_c = coeff_lookup(rows, p_coeffs)
    cols_is_p, cols_c = coeff_lookup(cols, p_coeffs)

    return ResidualBatch(
        h_rows_cols=jnp.asarray(H),
        x_rows=jnp.asarray(det_features(rows, ham.norb)),
        x_cols=jnp.asarray(det_features(cols, ham.norb)),
        rows_is_p=jnp.asarray(rows_is_p),
        cols_is_p=jnp.asarray(cols_is_p),
        rows_c=jnp.asarray(rows_c.astype(np.float32)),
        cols_c=jnp.asarray(cols_c.astype(np.float32)),
        energy=0.0,  # filled by caller
    )


def batch_loss(params, batch: ResidualBatch) -> jax.Array:
    amp_rows = mlp_apply(params, batch.x_rows)
    amp_cols = mlp_apply(params, batch.x_cols)

    psi_rows = jnp.where(batch.rows_is_p, batch.rows_c, amp_rows)
    psi_cols = jnp.where(batch.cols_is_p, batch.cols_c, amp_cols)

    hpsi_rows = batch.h_rows_cols @ psi_cols
    residual = hpsi_rows - batch.energy * psi_rows

    return jnp.mean(residual * residual)


value_and_grad = jax.value_and_grad(batch_loss)


# -----------------------------
# Training driver
# -----------------------------

def train_neural_external(
    ham,
    state,
    *,
    eps_pq: float | None = None,
    eps_qq: float = 1e-6,
    batch_size: int = 128,
    n_step: int = 500,
    hidden: int = 64,
    depth: int = 2,
    lr: float = 1e-3,
    seed: int = 7,
):
    """Train MLP external amplitudes against sampled Q residuals.

    This is intentionally minimal:
      - fixed P coefficients
      - fixed energy
      - no extra regularization
      - no handcrafted physical input features
      - no explicit H_eff matrix target
    """
    rng = np.random.default_rng(seed)

    p_dets = libdet.to_dets(state.dets)
    p_coeffs = np.asarray(state.coeffs, dtype=np.float64).reshape(-1)
    energy = float(state.energy)

    if eps_pq is None:
        eps_pq = float(state.eps if state.eps is not None else 1e-6)

    # Candidate Q space directly coupled to the HCI P space.
    q_pool = ham.expand(
        p_dets,
        eps_pq,
        coeffs=p_coeffs,
        exclude=p_dets,
    )

    q_pool = libdet.to_dets(q_pool)

    print(f"P size      : {len(p_dets)}")
    print(f"Q pool size : {len(q_pool)}")
    print(f"E fixed     : {energy:.12f}")
    print(f"eps_pq      : {eps_pq:.3e}")
    print(f"eps_qq      : {eps_qq:.3e}")
    print()

    key = jax.random.key(seed)
    params = init_mlp(key, din=2 * int(ham.norb), hidden=hidden, depth=depth)
    opt = adam_init(params)

    for step in range(1, n_step + 1):
        idx = rng.choice(
            len(q_pool),
            size=min(batch_size, len(q_pool)),
            replace=len(q_pool) < batch_size,
        )
        q_rows = q_pool[idx]

        batch = make_q_residual_batch(
            ham,
            p_dets,
            p_coeffs,
            q_rows,
            eps_qq=eps_qq,
        )
        batch.energy = energy

        loss, grads = value_and_grad(params, batch)
        params, opt = adam_step(params, grads, opt, lr=lr)

        if step == 1 or step % 25 == 0:
            print(f"{step:6d}  loss = {float(loss):.8e}")

    return params, q_pool


# -----------------------------
# Optional diagnostic
# -----------------------------

def estimate_projected_energy(
    ham,
    state,
    params,
    q_pool: np.ndarray,
    *,
    max_q: int = 4096,
) -> float:
    """Finite-pool diagnostic for c^T (H_PP c + H_PQ A_theta)."""
    p_dets = libdet.to_dets(state.dets)
    p_coeffs = np.asarray(state.coeffs, dtype=np.float64).reshape(-1)

    q_use = libdet.to_dets(q_pool[: min(max_q, len(q_pool))])
    cols = unique_dets(p_dets, q_use)

    p_coeffs_dict = {
        det_key(det): float(c)
        for det, c in zip(p_dets, p_coeffs)
    }

    cols_is_p, cols_c = coeff_lookup(cols, p_coeffs_dict)

    x_cols = jnp.asarray(det_features(cols, ham.norb))
    amp_cols = mlp_apply(params, x_cols)
    psi_cols = np.asarray(
        jnp.where(
            jnp.asarray(cols_is_p),
            jnp.asarray(cols_c.astype(np.float32)),
            amp_cols,
        ),
        dtype=np.float64,
    )

    hpsi_p = ham.matvec(p_dets, psi_cols, kets=cols)
    return float(np.dot(p_coeffs, hpsi_p) / np.dot(p_coeffs, p_coeffs))


# -----------------------------
# Example usage after your HCI run
# -----------------------------

# params, q_pool = train_neural_external(
#     ham,
#     state,
#     eps_pq=state.eps,
#     eps_qq=1e-6,
#     batch_size=128,
#     n_step=500,
#     hidden=64,
#     depth=2,
#     lr=1e-3,
#     seed=7,
# )
#
# e_proj = estimate_projected_energy(ham, state, params, q_pool)
# print(f"HCI var              : {state.energy:.12f}")
# print(f"Neural projected diag : {e_proj:.12f}")

In [5]:
def semi_pt2(
    ham,
    state: State,
    *,
    eps1: float = 1e-6,
    eps2: float = 1e-6,
    counts: int = 16,
    n_rep: int = 4,
    seed: int = 0,
) -> tuple[float, float, float, float]:

    dets = state.dets
    coeffs = state.coeffs
    energy = state.energy

    # Deterministic strong projected amplitudes at eps1.
    prj = ham.project(
        None,
        dets,
        coeffs,
        eps=eps1,
        exclude=dets,
    )
    hpsi = np.asarray(prj.hpsi, dtype=np.float64)
    diags = np.asarray(prj.diags, dtype=np.float64)

    e2_det = float(np.sum((hpsi * hpsi) / (energy - diags)))

    # Weak-window sampled projection in eps2 <= |H_ai c_i| < eps1.
    sample = ham.sample_project(
        dets,
        coeffs,
        eps1,
        eps2,
        counts,
        exclude=dets,
        n_rep=n_rep,
        seed=seed,
    )

    rep_ptr = np.asarray(sample.rep_ptr, dtype=np.int64)
    diags = np.asarray(sample.diags, dtype=np.float64)
    hpsi_strong = np.asarray(sample.hpsi_strong, dtype=np.float64)
    hpsi_a = np.asarray(sample.hpsi_a, dtype=np.float64)
    hpsi_b = np.asarray(sample.hpsi_b, dtype=np.float64)

    corr = np.zeros(n_rep, dtype=np.float64)
    for r in range(n_rep):
        lo = int(rep_ptr[r])
        hi = int(rep_ptr[r + 1])
        if hi == lo:
            continue

        denom = energy - diags[lo:hi]
        s = hpsi_strong[lo:hi]
        wa = hpsi_a[lo:hi]
        wb = hpsi_b[lo:hi]
        corr[r] = np.sum((s * (wa + wb) + wa * wb) / denom)

    e2_stoch = float(np.mean(corr))
    err = 0.0 if n_rep == 1 else float(np.std(corr, ddof=1) / np.sqrt(n_rep))
    return e2_det + e2_stoch, e2_det, e2_stoch, err


e2_total, e2_det, e2_stoch, err = semi_pt2(ham, state)
total_energy = state.energy + e2_total

print(f"Variational: {state.energy:16.12f}")
print(f"PT2: {e2_total:14.12f} +/- {err:14.12f}")
print(f"Total: {total_energy:16.12f} +/- {err:14.12f}")

Variational: -76.243207086118
PT2: -0.000561451458 +/- 0.000000000000
Total: -76.243768537576 +/- 0.000000000000
